# Utils

This notebook is only used to try out different things without changing the other notebooks too much.

In [7]:
import os
import shutil
import h5py
import numpy as np
import sys

sys.path.append("..")
sys.path.append("../code")

from dataset import PointCloudEmbeddingSequenceDataset
from models.DeepCAD.cadlib.visualize import vec2CADsolid
from OCC.Core.BRepCheck import BRepCheck_Analyzer
from OCC.Extend.DataExchange import write_step_file
from OCC.Core.STEPControl import STEPControl_Reader
from OCC.Core.StlAPI import StlAPI_Writer
from OCC.Core.BRepMesh import BRepMesh_IncrementalMesh

In [8]:
def copy_to_temp(dataset, idx):
    cad_seq_path = dataset.get_cad_seq_path(idx)
    pc_path = dataset.get_pc_path(idx)
    
    temp_dir = "../data/temporary"
    if os.path.exists(temp_dir):
        shutil.rmtree(temp_dir)
    os.makedirs(temp_dir)

    dest_path_cad_seq = os.path.join(temp_dir, os.path.basename(cad_seq_path))
    dest_path_pc = os.path.join(temp_dir, os.path.basename(pc_path))

    shutil.copy(cad_seq_path, dest_path_cad_seq)
    shutil.copy(pc_path, dest_path_pc)
    print(f"Copied {cad_seq_path} to {dest_path_cad_seq}")
    print(f"Copied {pc_path} to {dest_path_pc}")
    return dest_path_cad_seq

def change_keys(h5_file):
    """Changes the keys from 'vec' to 'out_vec' in order to be able to show the sample using show.py"""
    with h5py.File(h5_file, 'r+') as hf:
        if 'vec' in hf:
            data = hf['vec'][:]
            hf.create_dataset('out_vec', data=data)
            del hf['vec']
            print(f"Changed keys from 'vec' to 'out_vec' in {h5_file}")

def export2step(h5_path):
    filter = True
    save_path = os.path.join(*h5_path.split("/")[:-1], os.path.splitext(os.path.basename(h5_path))[0] + '.step')

    with h5py.File(h5_path, 'r') as fp:
        seq = fp['out_vec'][:].astype(np.float32)

        out_shape = vec2CADsolid(seq)

        if filter:
            analyzer = BRepCheck_Analyzer(out_shape)
            if not analyzer.IsValid():
                print(f"CAD-sequence of {os.path.basename(pc_path)} is invalid.")

        write_step_file(out_shape, save_path)
    return save_path

def step2stl(step_path):

    save_path = os.path.join(*step_path.split("/")[:-1], os.path.splitext(os.path.basename(step_path))[0] + '.stl')
    step_reader = STEPControl_Reader()
    step_reader.ReadFile(step_path)
    step_reader.TransferRoots()
    shape = step_reader.OneShape()

    BRepMesh_IncrementalMesh(shape, 0.1)

    stl_writer = StlAPI_Writer()
    stl_writer.Write(shape, save_path)
    print(f"Wrote stl file to {save_path}")

def visualize_gt(dataset, idx):
    dest_path_h5 = copy_to_temp(dataset, index)
    change_keys(dest_path_h5)
    step_path = export2step(dest_path_h5)
    step2stl(step_path)

In [15]:
index = 30149

In [16]:
dataset = PointCloudEmbeddingSequenceDataset("../data", 'train')

In [17]:
visualize_gt(dataset, index)

Copied ../data/cad_vec/0004/00041545.h5 to ../data/temporary/00041545.h5
Copied ../data/pc_cad/0004/00041545.ply to ../data/temporary/00041545.ply
Changed keys from 'vec' to 'out_vec' in ../data/temporary/00041545.h5

*******************************************************************
******        Statistics on Transfer (Write)                 ******
Wrote stl file to ../data/temporary/00041545.stl

*******************************************************************
******        Transfer Mode = 0  I.E.  As Is       ******
******        Transferring Shape, ShapeType = 2                      ******
** WorkSession : Sending all data
 Step File Name : ../data/temporary/00041545.step(696 ents)  Write  Done


In [1]:
import torch

In [30]:
lol = torch.load("../models/trained_models/github_model_pn/latest.pth", weights_only = True, map_location=torch.device('cpu'))

In [26]:
print(type(lol))
print(lol.keys())
for i, a in enumerate(lol['model_state_dict'].keys()):
    print(i, a.ljust(50), lol['model_state_dict'][a].shape)
print(lol['model_state_dict']['SA_modules.0.mlps.0.0.weight'])

<class 'dict'>
dict_keys(['clock', 'model_state_dict', 'optimizer_state_dict', 'scheduler_state_dict'])
0 SA_modules.0.mlps.0.0.weight                       torch.Size([32, 6, 1, 1])
1 SA_modules.0.mlps.0.1.weight                       torch.Size([32])
2 SA_modules.0.mlps.0.1.bias                         torch.Size([32])
3 SA_modules.0.mlps.0.1.running_mean                 torch.Size([32])
4 SA_modules.0.mlps.0.1.running_var                  torch.Size([32])
5 SA_modules.0.mlps.0.1.num_batches_tracked          torch.Size([])
6 SA_modules.0.mlps.0.3.weight                       torch.Size([32, 32, 1, 1])
7 SA_modules.0.mlps.0.4.weight                       torch.Size([32])
8 SA_modules.0.mlps.0.4.bias                         torch.Size([32])
9 SA_modules.0.mlps.0.4.running_mean                 torch.Size([32])
10 SA_modules.0.mlps.0.4.running_var                  torch.Size([32])
11 SA_modules.0.mlps.0.4.num_batches_tracked          torch.Size([])
12 SA_modules.0.mlps.0.6.weight        

In [27]:
lol['model_state_dict']["SA_modules.0.mlps.0.0.weight"] = lol['model_state_dict']["SA_modules.0.mlps.0.0.weight"][:, :3, :, :]

In [28]:
torch.save(lol, "../data/latent/pretrained/test_pretrained_pn2/pc2cad/model/latest_mod.pth")

In [31]:
lol = torch.load("../data/latent/pretrained/test_pretrained_pn2/pc2cad/model/latest_mod.pth", weights_only = True, map_location=torch.device('cpu'))

In [32]:
print(type(lol))
print(lol.keys())
for i, a in enumerate(lol['model_state_dict'].keys()):
    print(i, a.ljust(50), lol['model_state_dict'][a].shape)
print(lol['model_state_dict']['SA_modules.0.mlps.0.0.weight'])

<class 'dict'>
dict_keys(['clock', 'model_state_dict', 'optimizer_state_dict', 'scheduler_state_dict'])
0 SA_modules.0.mlps.0.0.weight                       torch.Size([32, 3, 1, 1])
1 SA_modules.0.mlps.0.1.weight                       torch.Size([32])
2 SA_modules.0.mlps.0.1.bias                         torch.Size([32])
3 SA_modules.0.mlps.0.1.running_mean                 torch.Size([32])
4 SA_modules.0.mlps.0.1.running_var                  torch.Size([32])
5 SA_modules.0.mlps.0.1.num_batches_tracked          torch.Size([])
6 SA_modules.0.mlps.0.3.weight                       torch.Size([32, 32, 1, 1])
7 SA_modules.0.mlps.0.4.weight                       torch.Size([32])
8 SA_modules.0.mlps.0.4.bias                         torch.Size([32])
9 SA_modules.0.mlps.0.4.running_mean                 torch.Size([32])
10 SA_modules.0.mlps.0.4.running_var                  torch.Size([32])
11 SA_modules.0.mlps.0.4.num_batches_tracked          torch.Size([])
12 SA_modules.0.mlps.0.6.weight        

In [33]:
lol = torch.load("../models/trained_models/fifth_official_run/best.pth", weights_only = True, map_location=torch.device('cpu'))

In [35]:
print(type(lol))
print(lol.keys())
for i, a in enumerate(lol['model_state_dict'].keys()):
    print(i, a.ljust(50), lol['model_state_dict'][a].shape)

<class 'dict'>
dict_keys(['model_state_dict', 'config'])
0 sa1.mlp_convs.0.weight                             torch.Size([64, 3, 1, 1])
1 sa1.mlp_convs.0.bias                               torch.Size([64])
2 sa1.mlp_convs.1.weight                             torch.Size([64, 64, 1, 1])
3 sa1.mlp_convs.1.bias                               torch.Size([64])
4 sa1.mlp_convs.2.weight                             torch.Size([128, 64, 1, 1])
5 sa1.mlp_convs.2.bias                               torch.Size([128])
6 sa1.mlp_bns.0.weight                               torch.Size([64])
7 sa1.mlp_bns.0.bias                                 torch.Size([64])
8 sa1.mlp_bns.0.running_mean                         torch.Size([64])
9 sa1.mlp_bns.0.running_var                          torch.Size([64])
10 sa1.mlp_bns.0.num_batches_tracked                  torch.Size([])
11 sa1.mlp_bns.1.weight                               torch.Size([64])
12 sa1.mlp_bns.1.bias                                 torch.Size([64])
1